In [ ]:
"""Generating an IBOR extract

Demonstrates how to use the GetHoldings API to generate IBOR extracts.

Attributes
----------
cocoon - seed_data
holdings
"""

# Generating an IBOR extract with LUSID's GetHoldings method

This notebooks shows how you can use the [GetHoldings](https://www.lusid.com/docs/api/#operation/GetHoldings) API to generate IBOR extracts.

## Setup LUSID

In [ ]:
# Import system packages

import os

# Import lusid specific packages
# These are the core lusid packages for interacting with the API via Python
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.seed_sample_data import seed_data
from finbourne_sdk_utils.cocoon.utilities import create_scope_id

# Import data wrangling and data management packages
import pandas as pd
import numpy as np
import json
import openpyxl
pd.set_option('display.max_columns', None)

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook")

In [ ]:
# Load a mapping file for formatting DataFrame columns

with open(r"config/build_transactions_mapping.json") as mappings_file:
    build_txn_json_mappings = json.load(mappings_file)

## Load transactions into a new scope

For the purpose of this demo, we first need to make sure we have a portfolio with some data. In the code below, we create a new portfolio called <b>EQUITY_UK</b> with some transactions from the <i>equity_transactions.csv</i> file. The focus of this notebook is on getting holdings so we use the <i>seed_data()</i> function to quickly generate a demo portfolio. For a more in-depth look into loading transactions and an instrument master in LUSID, see our long-form [tutorial](https://support.finbourne.com/how-do-i-create-holdings) on the support page. 

In [ ]:
# Create a new scope
# Declare a variable to hold our portfolio name

scope = create_scope_id()
portfolio_code = "EQUITY_UK"

In [ ]:
# Load a file of equity transactions

transactions_file = r"data/equity_transactions.csv"
transactions_df = pd.read_csv(transactions_file)
transactions_df["portfolio_code"] = portfolio_code

In [ ]:
# The seed_data() function takes a file of transaction data
# and loads portfolios, instruments, and transactions into LUSID
# We use this function as a quick way of generating a demo portfolio

seed_data_response = seed_data(api_factory,
          ["portfolios", "instruments", "transactions"],
          scope,
          transactions_file,
          "csv"
         )

## GetHoldings

We call the <b>GetHoldings</b> method to generate an IBOR extract. This method requires two inputs: a portfolio and a scope. If we do not pass a date, then the default position date is today. You can modify the <b>scope</b> and <b>code</b> below to generate holdings for your own portfolios.

In [ ]:
# Define the transaction portfolio API

txn_port_api = api_factory.build(lu.TransactionPortfoliosApi)

In [ ]:
# Call the get_holdings method on the TransactionPortfoliosApi object
# The python SDK uses snake case get_holdings to represent LUSID's GetHoldings method
# We also pass a property to show each instrument's name

holdings_response = txn_port_api.get_holdings(scope=scope,
                                     code=portfolio_code,
                                     property_keys=["Instrument/default/Name"])

In the holdings's response, we can see a seperate value (represented as a row in the DataFrame) for each holdings. Holdings in securities have a <b>HoldingType</b> of <b>P</b> while holdings of cash balances have a <b>HoldingType</b> of <b>B</b>. See the linked [tutorial](https://support.finbourne.com/how-do-i-create-holdings) for a full list of holding types in LUSID.

In [ ]:
# Convert JSON response to DataFrame

lusid_response_to_data_frame(holdings_response, rename_properties=True).head(10)

## GetHoldings for a specific date and time

In the code below, we generate holdings again, but this time we provide an effective date of 5 January 20202 at 12:30 PM (UTC). As we can see in the DataFrame below, the portfolio has a different set of holdings on that date.

If you pass a date with no time (e.g. <b>2020-01-05</b> then LUSID will set the default time to midnight or <b>00:00:00</b>)

In [ ]:

holdings_response_fifth_jan = txn_port_api.get_holdings(scope=scope,
                                     code=portfolio_code,
                                     effective_at="2020-01-05T12:30:00Z",
                                     property_keys=["Instrument/default/Name"])

In [ ]:
# Convert JSON response to DataFrame

holdings_fifth_jan = lusid_response_to_data_frame(holdings_response_fifth_jan, rename_properties=True)

#Format the columns we want to view

column_rename = {
    "instrument_uid": "luid",
    "Name(default-Properties)": "instrumentName",
    "holding_type": "holdingType",
    "units": "units",
    "settled_units": "settledUnits",
    "cost.amount": "costAmount",
    "cost.currency": "costCurrency",
    "cost_portfolio_ccy.currency": "portfolioCurrency"
}

holdings_fifth_jan = holdings_fifth_jan.rename(columns = column_rename)[column_rename.values()].copy()
display(holdings_fifth_jan)

## What transactions produced my holdings?

The holdings in LUSID's <b>Transaction Portfolios</b> are produced as the result of transactions. You can link a holding back to its source transaction using the build transactions endpoint. Let's do a deep-dive into <i>Tesco</i> which is one of our holdings.

In [ ]:
# Call the build transactions endpoint

build_transactions_response = txn_port_api.build_transactions(scope=scope,
                               code=portfolio_code,
                                property_keys=["Instrument/default/Name"],
                               transaction_query_parameters = models.TransactionQueryParameters(start_date="2020-01-01",
                                                                                   end_date="2020-12-31"))

By calling the build transactions endpoint, we can see that two trades produce our holding of 12,000 units:

* One transaction on the 17 January 2020 for 8,000 units
* Then a second transaction on the 18 January 2020 for 4,000 units

The <b>ResultantHolding</b> column tracks the total holding in the portfolio for each unique instrument as of the transaction effective date and time. 

In [ ]:
# Filter for a list of columns we want to show in the notebook

columns = ["TransactionId",
"TransactionType",
"LusidInstrumentId",
"InstrumentName",
"SettlementDate",
"Price",
"Units",
"ResultantHolding"]

# Target the rows for Tesco only

transactions_df = lusid_response_to_data_frame(build_transactions_response, column_name_mapping=build_txn_json_mappings, use_camel_case=True)
transactions_df = transactions_df[columns]
transactions_df

## Exporting holdings into an Excel file

You can then use standard panda's functions to export the holdings into an Excel file (or various other formats required by downstream systems or users).

In [ ]:
holdings_fifth_jan.to_excel("data/extracts/holdings_20200105.xlsx")